In [3]:
import plotly.io as pio
pio.renderers.default = 'notebook'

In [33]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, auc, accuracy_score, average_precision_score
import numpy as np
import random
import pandas as pd
import time
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================
# UTILS & MODEL COUNTING
# ==========================================

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device:",device)

# ==========================================
# DATA LOADING
# ==========================================

data = torch.load("../wildfire_dataset.pth") 
X, y = data['X'], data['y']  

num_features = X.shape[2]
set_seed(42) 

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=128, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test, y_test), batch_size=128, shuffle=False)

pos_weight = ((y_train == 0).sum() / (y_train == 1).sum().clamp(min=1)).clone().detach().to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# ==========================================
# MODEL DEFINITIONS (7 Variants)
# ==========================================

class CNN_BiLSTM(nn.Module):
    def __init__(self, input_features):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(input_features, 32, 3, padding=1), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool1d(2)
        )
        self.lstm = nn.LSTM(64, 64, num_layers=2, dropout=0.3, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(128, 1)

    def forward(self, x):
        x = self.cnn(x.permute(0, 2, 1)).permute(0, 2, 1)
        _, (hn, _) = self.lstm(x)
        out = torch.cat((hn[-2,:,:], hn[-1,:,:]), dim=1) 
        return self.fc(out).squeeze(1)

class FCN_Model(nn.Module):
    def __init__(self, input_features):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv1d(input_features, 128, 8, padding='same'), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 256, 5, padding='same'), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Conv1d(256, 128, 3, padding='same'), nn.BatchNorm1d(128), nn.ReLU()
        )
        self.fc = nn.Linear(128, 1)

    def forward(self, x):
        x = self.convs(x.permute(0, 2, 1))
        x = x.mean(dim=2) 
        return self.fc(x).squeeze(1)

class ResBlock1D(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm1d(channels)
        self.conv2 = nn.Conv1d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm1d(channels)

    def forward(self, x):
        res = x
        x = torch.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return torch.relu(x + res)

class ResNet_LSTM(nn.Module):
    def __init__(self, input_features):
        super().__init__()
        self.entry = nn.Conv1d(input_features, 64, 1)
        self.res1 = ResBlock1D(64)
        self.res2 = ResBlock1D(64)
        self.pool = nn.MaxPool1d(2)
        self.lstm = nn.LSTM(64, 64, batch_first=True)
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        x = self.entry(x.permute(0, 2, 1))
        x = self.res2(self.res1(x))
        x = self.pool(x).permute(0, 2, 1)
        _, (hn, _) = self.lstm(x)
        return self.fc(hn[-1]).squeeze(1)

class Transformer_Model(nn.Module):
    def __init__(self, input_features, d_model=64):
        super().__init__()
        self.proj = nn.Linear(input_features, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=4, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=3)
        self.fc = nn.Linear(d_model, 1)

    def forward(self, x):
        x = self.proj(x)
        x = self.transformer(x)
        out = x.mean(dim=1) 
        return self.fc(out).squeeze(1)

class Simple_LSTM(nn.Module):
    def __init__(self, input_features):
        super().__init__()
        self.lstm = nn.LSTM(input_features, 64, num_layers=2, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        return self.fc(hn[-1]).squeeze(1)

class Simple_GRU(nn.Module):
    def __init__(self, input_features):
        super().__init__()
        self.gru = nn.GRU(input_features, 64, num_layers=2, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        _, hn = self.gru(x)
        return self.fc(hn[-1]).squeeze(1)

class Query_Attention(nn.Module):
    def __init__(self, input_features, embed_dim=64):
        super().__init__()
        self.proj = nn.Linear(input_features, embed_dim)
        self.query = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.attention = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=4, batch_first=True)
        self.fc = nn.Linear(embed_dim, 1)

    def forward(self, x):
        x = self.proj(x)
        batch_size = x.size(0)
        q = self.query.repeat(batch_size, 1, 1)
        attn_out, _ = self.attention(q, x, x)
        return self.fc(attn_out.squeeze(1)).squeeze(1)


# ==========================================
# TRAINING PIPELINE
# ==========================================

def train_and_evaluate(model, model_name, epochs=70, patience=10):
    print(f"--- Training {model_name} ---")
    model.to(device)
    param_count = count_parameters(model)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min',      
    factor=0.5,      
    patience=3,      
    verbose=True     
)
    best_val_loss = np.inf
    counter = 0  
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    start_time = time.time()
    for epoch in range(epochs):
        model.train()
        t_loss, t_correct, t_total = 0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device).float()
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            t_loss += loss.item()
            t_correct += ((preds > 0) == yb).sum().item()
            t_total += yb.size(0)
        
        # Validation
        model.eval()
        v_loss, v_correct, v_total = 0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device).float()
                preds = model(xb)
                v_loss += criterion(preds, yb).item()
                v_correct += ((preds > 0) == yb).sum().item()
                v_total += yb.size(0)
        
        scheduler.step(v_loss / len(val_loader))
        
        history['train_loss'].append(t_loss/len(train_loader))
        history['val_loss'].append(v_loss/len(val_loader))
        history['train_acc'].append(t_correct/t_total)
        history['val_acc'].append(v_correct/v_total)

        if v_loss < best_val_loss:
            best_val_loss = v_loss
            torch.save(model.state_dict(), "temp.pth")
            counter = 0
        else:
            counter += 1
            if counter >= patience: break

    train_time = time.time() - start_time
    model.load_state_dict(torch.load("temp.pth", weights_only=True))
    model.eval()
    
    # Testing
    all_preds, all_targets = [], []
    inf_start = time.time()
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            all_preds.append(torch.sigmoid(model(xb)).cpu())
            all_targets.append(yb.cpu())
    inf_time = ((time.time() - inf_start) / len(X_test)) * 1000

    all_preds, all_targets = torch.cat(all_preds).numpy(), torch.cat(all_targets).numpy()
    
    metrics = {
        "Model": model_name,
        "Parameters": param_count,
        "Accuracy": accuracy_score(all_targets, (all_preds > 0.5)),
        "PR-AUC": auc(*precision_recall_curve(all_targets, all_preds)[1::-1]),
        "AP": average_precision_score(all_targets, all_preds),
        "Inference (ms)": inf_time,
        "Train Time (s)": train_time
    }
    return metrics, history

# ==========================================
# EXECUTION & ADVANCED PLOTTING
# ==========================================

models_list = [CNN_BiLSTM, FCN_Model, ResNet_LSTM, Transformer_Model, Simple_LSTM, Simple_GRU, Query_Attention]
names_list = ["CNN_BiLSTM", "FCN", "ResNet_LSTM", "Transformer", "LSTM", "GRU", "Query_Attention"]

results = []
all_histories = {}

for cls, name in zip(models_list, names_list):
    m, h = train_and_evaluate(cls(num_features), name)
    results.append(m)
    all_histories[name] = h

device: cuda
--- Training CNN_BiLSTM ---
Epoch 00028: reducing learning rate of group 0 to 5.0000e-05.
Epoch 00041: reducing learning rate of group 0 to 2.5000e-05.
Epoch 00045: reducing learning rate of group 0 to 1.2500e-05.
Epoch 00054: reducing learning rate of group 0 to 6.2500e-06.
Epoch 00060: reducing learning rate of group 0 to 3.1250e-06.
Epoch 00064: reducing learning rate of group 0 to 1.5625e-06.
--- Training FCN ---
Epoch 00028: reducing learning rate of group 0 to 5.0000e-05.
Epoch 00036: reducing learning rate of group 0 to 2.5000e-05.
Epoch 00040: reducing learning rate of group 0 to 1.2500e-05.
Epoch 00045: reducing learning rate of group 0 to 6.2500e-06.
Epoch 00053: reducing learning rate of group 0 to 3.1250e-06.
Epoch 00057: reducing learning rate of group 0 to 1.5625e-06.
--- Training ResNet_LSTM ---
Epoch 00022: reducing learning rate of group 0 to 5.0000e-05.
Epoch 00044: reducing learning rate of group 0 to 2.5000e-05.
Epoch 00053: reducing learning rate of gr

In [34]:
import plotly.io as pio
import plotly.graph_objects as go

# 4. ARCHITECTURE EFFICIENCY (Fixed Bubbles & Log Scale)
fig_efficiency = go.Figure()
max_inf = df['Inference (ms)'].max()

for i, row in df.iterrows():
    fig_efficiency.add_trace(go.Scatter(
        x=[row['Parameters']], 
        y=[row['PR-AUC']], 
        mode='markers+text',
        name=row['Model'],
        text=[row['Model']],
        textposition="top center",
        marker=dict(
            size=[row['Inference (ms)']],
            sizemode='area',
            sizeref=2. * max_inf / (50.**2), 
            sizemin=12,
            color=colors[i % len(colors)],
            line=dict(width=1.5, color='rgba(50, 50, 50, 0.8)')
        )
    ))

fig_efficiency.update_layout(
    title="Computational Efficiency vs. Predictive Power",
    xaxis_title="Number of Trainable Parameters (Log Scale)",
    yaxis_title="Precision-Recall AUC",
    xaxis=dict(type="log", gridcolor='lightgrey'),
    yaxis=dict(gridcolor='lightgrey'),
    font=paper_font,
    width=950, height=650,
    legend=dict(x=1.02, y=1, borderwidth=1)
)
fig_efficiency.show()

# 5. RADAR CHART (Simplified for Comparison)

colors = ['#E41A1C', '#377EB8', '#4DAF4A', '#984EA3', '#FF7F00']

fig_radar = go.Figure()
metrics_radar = ['Accuracy', 'PR-AUC', 'AP']

for i, row in df.iterrows():
    fig_radar.add_trace(go.Scatterpolar(
        r=[row[m] for m in metrics_radar] + [row[metrics_radar[0]]],
        theta=metrics_radar + [metrics_radar[0]],
        name=row['Model'],
        line=dict(width=3, color=colors[i % len(colors)]),
        marker=dict(size=8),
        fill='toself',
        fillcolor=f"rgba{tuple(list(int(colors[i % len(colors)][j:j+2], 16) for j in (1, 3, 5)) + [0.1])}"
    ))

fig_radar.update_layout(
    polar=dict(
        bgcolor="white", # Clean white background
        radialaxis=dict(
            visible=True,
            range=[0.2, 0.8],
            gridcolor="#D3D3D3", # Subdued grid
            gridwidth=1,
            tickvals=[0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
            tickfont=dict(size=14, color="black", family="serif")
        ),
        angularaxis=dict(
            gridcolor="#D3D3D3",
            rotation=90, # Align first metric to the top
            direction="clockwise",
            tickfont=dict(size=16, color="black", family="serif")
        )
    ),
    # Use 'serif' to match LaTeX/Times New Roman academic fonts
    font=dict(family="serif", size=14, color="black"),
    title=dict(
        text="<b>Figure 1: Comparison of Model Performance Metrics</b>",
        x=0.5,
        y=0.95,
        font=dict(size=20)
    ),
    width=800,
    height=700,
    margin=dict(l=80, r=80, t=100, b=80),
    legend=dict(
        orientation="h",   # Horizontal legend is often better for paper layouts
        yanchor="bottom",
        y=-0.2,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(255,255,255,0)",
        bordercolor="black",
        borderwidth=1
    ),
    paper_bgcolor="white",
    plot_bgcolor="white"
)

# To save for a paper, always export to a vector format (PDF or EPS)
# fig_radar.write_image("radar_chart.pdf")

fig_radar.show()

In [35]:
import plotly.graph_objects as go
import plotly.io as pio

# 1. SETUP: Academic Standards
pio.templates.default = "plotly_white"
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2']
paper_font = dict(family="Arial", size=18, color="black")

def create_paper_plot(title, y_label):
    """Helper to create a consistent, clean figure layout."""
    fig = go.Figure()
    fig.update_layout(
        title=dict(text=title, font=dict(size=24)),
        xaxis_title="Epochs",
        yaxis_title=y_label,
        font=paper_font,
        width=1000,
        height=600,
        legend=dict(x=1.02, y=1, xanchor='left', yanchor='top', borderwidth=1),
        margin=dict(l=80, r=180, t=100, b=80)
    )
    # Professional axis lines
    fig.update_xaxes(showline=True, linewidth=1.5, linecolor='black', gridcolor='lightgrey')
    fig.update_yaxes(showline=True, linewidth=1.5, linecolor='black', gridcolor='lightgrey')
    return fig

# 2. GENERATING THE 4 INDIVIDUAL PLOTS

# --- Plot 1: Training Loss ---
fig1 = create_paper_plot("Figure 1: Training Convergence (BCE Loss)", "BCE Loss")
for i, (name, h) in enumerate(all_histories.items()):
    fig1.add_trace(go.Scatter(x=list(range(len(h['train_loss']))), y=h['train_loss'], 
                             name=name, line=dict(color=colors[i % len(colors)], width=2), opacity=0.8))
fig1.show()

# --- Plot 2: Validation Loss ---
fig2 = create_paper_plot("Figure 2: Validation Generalization (BCE Loss)", "BCE Loss")
for i, (name, h) in enumerate(all_histories.items()):
    fig2.add_trace(go.Scatter(x=list(range(len(h['val_loss']))), y=h['val_loss'], 
                             name=name, line=dict(color=colors[i % len(colors)], width=2), opacity=0.8))
fig2.show()

# --- Plot 3: Training Accuracy ---
fig3 = create_paper_plot("Figure 3: Training Classification Accuracy", "Accuracy")
for i, (name, h) in enumerate(all_histories.items()):
    fig3.add_trace(go.Scatter(x=list(range(len(h['train_acc']))), y=h['train_acc'], 
                             name=name, line=dict(color=colors[i % len(colors)], width=2), opacity=0.8))
fig3.show()

# --- Plot 4: Validation Accuracy ---
fig4 = create_paper_plot("Figure 4: Validation Classification Accuracy", "Accuracy")
for i, (name, h) in enumerate(all_histories.items()):
    fig4.add_trace(go.Scatter(x=list(range(len(h['val_acc']))), y=h['val_acc'], 
                             name=name, line=dict(color=colors[i % len(colors)], width=2), opacity=0.8))
fig4.show()

In [37]:
print(df.to_string(index=False))

          Model  Parameters  Accuracy  PR-AUC     AP  Inference (ms)  Train Time (s)
            FCN      284289    0.7095  0.5981 0.5982          0.0157        316.0384
    Transformer      844865    0.6668  0.5442 0.5443          0.0293        606.5281
    ResNet_LSTM       84609    0.6858  0.5256 0.5258          0.0235        559.9415
     CNN_BiLSTM      174177    0.6771  0.5153 0.5155          0.0172        389.7276
            GRU       41537    0.6485  0.5087 0.5089          0.0127        362.9160
           LSTM       55361    0.6681  0.5028 0.5030          0.0134        443.4892
Query_Attention       18113    0.6171  0.4148 0.4151          0.0124        269.8537
